# CMSC 173 &middot; Machine Learning &mdash; Week 13 Lab
## Neural Networks: A 2-Layer Net from Scratch

A neural network is just linear layers with a squish in between &mdash; and that squish is what lets
it bend. You'll build a **2-layer network** in pure NumPy: the **forward pass** that makes a
prediction, and **backpropagation** (the chain rule) that trains it &mdash; then watch it carve a
**curved boundary** no straight line could.

**How this lab works.** Each part = a short **plain-English explainer**, a **code cell**
you run, a **line-by-line walkthrough** of what it did, and an **Answer here** box. The
code does the maths; we *graph* the results so you can see what is going on.

**NumPy + Matplotlib (from scratch).** **Not graded.** About 60 minutes.

---
## Part 0 &middot; Setup + data that curves

`make_moons` gives two interleaving crescents &mdash; impossible for a straight line, easy for a network.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from sklearn.datasets import make_moons
rng = np.random.default_rng(173)

X, y = make_moons(n_samples=300, noise=0.2, random_state=173)
y = y.reshape(-1, 1)                                   # column vector, shape (300, 1)

def show(ax):
    ax.scatter(X[y[:,0]==0,0], X[y[:,0]==0,1], marker='o', alpha=0.6, label='class 0')
    ax.scatter(X[y[:,0]==1,0], X[y[:,0]==1,1], marker='^', alpha=0.6, label='class 1')
plt.figure(figsize=(6,5)); show(plt.gca()); plt.legend(); plt.title('two moons'); plt.tight_layout(); plt.show()

**Reading the code:** two crescent-shaped classes that interlock. No line separates them &mdash; you
need a curved boundary, which is exactly what a hidden layer provides.

---
## Part 1 &middot; The architecture & the forward pass

Our net: 2 inputs &rarr; a **hidden layer** of `H` units with a `tanh` squish &rarr; 1 output with a sigmoid
(a probability). The forward pass is two linear steps with a squish between:

$$a_1 = \tanh(XW_1 + b_1), \qquad \hat p = \sigma(a_1 W_2 + b_2).$$

Without the `tanh`, two stacked linear layers collapse into one line &mdash; the squish is the magic.

In [ ]:
H = 8                                                 # hidden units
W1 = rng.normal(0, 0.5, (2, H)); b1 = np.zeros(H)     # layer 1 weights & bias
W2 = rng.normal(0, 0.5, (H, 1)); b2 = np.zeros(1)     # layer 2 weights & bias

def sigmoid(z): return 1 / (1 + np.exp(-z))

def forward(X, W1, b1, W2, b2):
    z1 = X @ W1 + b1                                   # (1) linear layer 1
    a1 = np.tanh(z1)                                   # (2) the squish (non-linearity)
    z2 = a1 @ W2 + b2                                  # (3) linear layer 2
    p  = sigmoid(z2)                                   # (4) squash to a probability
    return p, a1

p, _ = forward(X, W1, b1, W2, b2)
print('output shape:', p.shape, '| a random untrained prediction:', round(float(p[0]), 3))

**Reading the code, line by line:**
- **(1)** `X @ W1 + b1` mixes the 2 inputs into `H` hidden scores &mdash; a linear step.
- **(2)** `np.tanh` bends each hidden score into (&minus;1, 1). **This** is what makes the network more
  than a line.
- **(3)** a second linear step combines the hidden units into one score.
- **(4)** the sigmoid turns that score into a probability. With random weights the prediction is
  meaningless &mdash; training will fix that.

---
## Part 2 &middot; Backpropagation & training

To train, we need the gradient of the loss for **every** weight. Backprop is just the **chain rule**
applied layer by layer, from the output backwards. The output error is `p - y`; we pass it back
through `W2`, undo the `tanh` (its derivative is `1 - a1**2`), and reach `W1`.

In [ ]:
lr, epochs = 0.5, 3000
m = len(X)
history = []
for _ in range(epochs):
    p, a1 = forward(X, W1, b1, W2, b2)                 # forward pass
    loss = -np.mean(y*np.log(p+1e-9) + (1-y)*np.log(1-p+1e-9))
    history.append(loss)
    # ---- backprop (chain rule, output -> input) ----
    dz2 = (p - y) / m                                  # (1) output-layer error
    dW2 = a1.T @ dz2;   db2 = dz2.sum(axis=0)          # (2) grads for layer 2
    da1 = dz2 @ W2.T                                   # (3) push error back to the hidden layer
    dz1 = da1 * (1 - a1**2)                            # (4) undo the tanh (its derivative)
    dW1 = X.T @ dz1;    db1 = dz1.sum(axis=0)          # (5) grads for layer 1
    # ---- gradient-descent step ----
    W1 -= lr*dW1; b1 -= lr*db1; W2 -= lr*dW2; b2 -= lr*db2

plt.figure(figsize=(6,3.5)); plt.plot(history)
plt.xlabel('epoch'); plt.ylabel('cross-entropy loss'); plt.title('Training loss'); plt.tight_layout(); plt.show()
acc = np.mean((forward(X, W1, b1, W2, b2)[0] >= 0.5) == y)
print(f'training accuracy after training: {acc:.3f}')

**Reading the code, line by line:**
- **(1)** `dz2 = (p - y)/m` &mdash; the error at the output (same clean form as logistic regression).
- **(2)** the layer-2 gradients: multiply that error by the hidden activations.
- **(3)** `dz2 @ W2.T` sends the error *backwards* into the hidden layer &mdash; this is the
  'back-propagation' step.
- **(4)** `* (1 - a1**2)` undoes the `tanh` (that's its derivative); the error is now at `z1`.
- **(5)** the layer-1 gradients, then all four parameters step downhill. The loss curve dives and
  accuracy climbs well past what any line could reach.

**Answer here:**

1. Step (4) multiplies by `1 - a1**2`, the derivative of `tanh`. In one sentence, what is backprop
   fundamentally doing when it goes layer by layer like this?
   &rarr; *your answer*

---
## Part 3 &middot; The curved boundary it learned

Colour the plane by the network's prediction. Unlike every model before the hidden layer, this
boundary can **bend** to trace the moons.

In [ ]:
xx, yy = np.meshgrid(np.linspace(X[:,0].min()-.5, X[:,0].max()+.5, 300),
                     np.linspace(X[:,1].min()-.5, X[:,1].max()+.5, 300))
grid = np.column_stack([xx.ravel(), yy.ravel()])
Z = forward(grid, W1, b1, W2, b2)[0].reshape(xx.shape)

plt.figure(figsize=(6.5,5))
plt.contourf(xx, yy, Z, levels=20, cmap='RdBu_r', alpha=0.6)
plt.contour(xx, yy, Z, levels=[0.5], colors='k')
show(plt.gca()); plt.legend(); plt.title('A neural net bends the boundary around the moons')
plt.tight_layout(); plt.show()

**Reading the code:** same grid-and-shade trick as logistic regression, but the black 0.5 contour is
now **curved**, wrapping around each crescent. That curvature came entirely from the 8 `tanh` hidden
units &mdash; take them away (or the `tanh`) and you'd be back to a straight line.

**Answer here:**

1. Set `H = 1` (one hidden unit) in Part 1, re-run Parts 1-3, and look at the boundary. Why does a
   tiny hidden layer struggle, and what does adding units buy you?
   &rarr; *your answer*

---
## Where you actually are

Set the pace honestly. Replace each `-` with: **solid** / **rusty** / **never really got it**.

| | You |
|---|---|
| Why a hidden layer needs a non-linearity | - |
| The forward pass (two linear steps + squish) | - |
| Backprop as the chain rule | - |
| Reading a training-loss curve | - |
| A curved decision boundary | - |

**Which part took longest, and where did you get stuck?**
&rarr; *your answer*

**In one plain sentence: what does the hidden layer's activation function make possible?**
&rarr; *your answer*

---
## Stretch &mdash; optional

Required part is done; nothing below is graded.

### Stretch &middot; ReLU instead of tanh

Modern nets often use **ReLU** (`max(0, z)`) in the hidden layer. Its derivative is `1` where `z>0`
else `0`. Change `np.tanh(z1)` to `np.maximum(0, z1)` in `forward`, and in backprop change
`(1 - a1**2)` to `(z1 > 0)`. Does it still learn the moons? Try it below (copy the two cells).

In [ ]:
# your code here (optional): rebuild forward() and the training loop with ReLU, report accuracy


---
## Submitting

Run the cell below. It uploads this notebook straight from Colab &mdash; nothing to download.

You need a **submit token**: open
[https://portal.latarak.com/student/submit-token](https://portal.latarak.com/student/submit-token),
sign in, press the button, then paste it when the cell asks. The cell hides what you type.

In [ ]:
# --- Submit this notebook ------------------------------------------------------
# Colab only. Anywhere else, use the manual route described below this cell.
import getpass, json, urllib.request, urllib.error

PORTAL, COURSE, WEEK = "https://portal.latarak.com", "cmsc173", 13

try:
    from google.colab import _message
except ImportError:
    raise SystemExit(
        "Not running in Colab. Download this notebook "
        "(File > Download > Download .ipynb) and upload it at "
        "https://portal.latarak.com/course/cmsc173/lab/13/submit"
    )

nb = _message.blocking_request("get_ipynb", timeout_sec=90)["ipynb"]
token = getpass.getpass("Submit token (hidden as you type): ").strip()

req = urllib.request.Request(
    PORTAL + "/api/labs/" + COURSE + "/submit-notebook",
    data=json.dumps({"week": WEEK, "notebook": nb}).encode(),
    headers={"Content-Type": "application/json", "Authorization": "Bearer " + token},
    method="POST",
)
try:
    with urllib.request.urlopen(req, timeout=120) as r:
        out = json.load(r)
    print("Submitted", out["course"], "week", out["week"], "for", out["student"])
    print(out["cells"], "cells,", out["executed"], "executed")
    print(out["message"])
except urllib.error.HTTPError as e:
    print("Not submitted:", json.loads(e.read()).get("error", e.reason))

Prefer to do it by hand? **File &rarr; Download &rarr; Download .ipynb**, then go to the
[Week 13 submission page](https://portal.latarak.com/course/cmsc173/lab/13/submit) and upload it.

Blank cells are fine and guesses are fine. Don't polish this until it hides what you knew.